# External linear / diffusion / spectral baselines (rebuttal to Con 3)

Reviewer asks for more comparisons with existing linear, diffusion-based, and spectral graph embedding methods. We add one **parameter-free**, **inductive**, **attributed** baseline per bucket and evaluate under the same strict-inductive protocol as GW-CCA.

| bucket | method | operator applied to $X$ | citation |
|---|---|---|---|
| linear | **SGC-lin** | $Z = \hat S^{K} X$, $K{=}3$ | Wu et al., ICML 2019 |
| diffusion | **APPNP-lin** | $K$-step PPR power iteration on $X$, $K{=}10, \alpha{=}0.1$ | Klicpera et al., ICLR 2019 |
| spectral | **Linear ChebNet** | $\text{concat}_{k=0..K}\, T_k(\tilde L)\,X$, $K{=}3$ | Defferrard et al., NeurIPS 2016 |

All three are the linear-head variants (no MLP / no ReLU). Embeddings are L2-row-normalised and fed to the same LBFGS Hadamard-product LP probe / linear NC probe as GW-CCA.

## How to reproduce

```bash
cd experiments/m_ablation
# link prediction
python run_linear_baselines.py --dataset ogbn-arxiv     --tasks lp --out ogbn-arxiv_lb.json
python run_linear_baselines.py --dataset reddit          --tasks lp --out reddit_lb.json
python run_linear_baselines.py --dataset yelp            --tasks lp --out yelp_lb.json
python run_linear_baselines.py --dataset ogbn-products   --tasks lp --methods sgc,appnp --out ogbn-products_lb.json
python run_linear_baselines.py --dataset ppi             --tasks lp --out ppi_lb.json
# node classification
python run_linear_baselines.py --dataset ogbn-arxiv     --tasks nc --out ogbn-arxiv_nc_lb.json
python run_linear_baselines.py --dataset reddit          --tasks nc --out reddit_nc_lb.json
python run_linear_baselines.py --dataset yelp            --tasks nc --out yelp_nc_lb.json
python run_linear_baselines.py --dataset ogbn-products   --tasks nc --methods sgc --out ogbn-products_nc_lb.json
python run_linear_baselines.py --dataset ppi             --tasks nc --out ppi_nc_lb.json
```

In [1]:
import json

DATASETS = ["ogbn-arxiv", "reddit", "yelp", "ogbn-products", "ppi"]
METHODS = ["sgc", "appnp", "chebnet"]


def _load_lp(ds):
    with open(f"{ds}_lb.json") as f:
        return {r["method"]: r for r in json.load(f)}


def _load_nc(ds):
    try:
        with open(f"{ds}_nc_lb.json") as f:
            return {r["method"]: r for r in json.load(f)}
    except FileNotFoundError:
        return {}


def _load_gcca(ds):
    with open(f"../../results/_res_GCCA__{ds}.json") as f:
        return json.load(f)


LP = {ds: _load_lp(ds) for ds in DATASETS}
NC = {ds: _load_nc(ds) for ds in DATASETS}
GCCA = {ds: _load_gcca(ds) for ds in DATASETS}
print("loaded", len(DATASETS), "datasets")

loaded 5 datasets


## LP AUC (%)

In [2]:
import pandas as pd


def _lp_auc(rec):
    return rec.get("auc", rec.get("lp_auc", float("nan")))


rows = {
    "GW-CCA (p1p2)": [GCCA[ds]["auc"] * 100 for ds in DATASETS],
    "SGC": [_lp_auc(LP[ds].get("sgc", {})) * 100 if LP[ds].get("sgc") else None for ds in DATASETS],
    "APPNP-lin": [_lp_auc(LP[ds].get("appnp", {})) * 100 if LP[ds].get("appnp") else None for ds in DATASETS],
    "Linear ChebNet": [_lp_auc(LP[ds].get("chebnet", {})) * 100 if LP[ds].get("chebnet") else None for ds in DATASETS],
}
auc = pd.DataFrame(rows, index=DATASETS)
auc.index.name = "dataset"
auc.round(2)

,GW-CCA (p1p2),SGC,APPNP-lin,Linear ChebNet
dataset,,,,
ogbn-arxiv,97.50,93.03,88.42,92.84
reddit,97.56,88.16,85.03,83.74
yelp,93.75,81.64,76.61,90.71
ogbn-products,99.55,99.61,99.57,NaN
ppi,89.21,85.27,72.77,67.41


## LP AP (%)

In [3]:
def _lp_ap(rec):
    return rec.get("ap", rec.get("lp_ap", float("nan")))


rows = {
    "GW-CCA (p1p2)": [GCCA[ds]["ap"] * 100 for ds in DATASETS],
    "SGC": [_lp_ap(LP[ds].get("sgc", {})) * 100 if LP[ds].get("sgc") else None for ds in DATASETS],
    "APPNP-lin": [_lp_ap(LP[ds].get("appnp", {})) * 100 if LP[ds].get("appnp") else None for ds in DATASETS],
    "Linear ChebNet": [_lp_ap(LP[ds].get("chebnet", {})) * 100 if LP[ds].get("chebnet") else None for ds in DATASETS],
}
ap = pd.DataFrame(rows, index=DATASETS)
ap.index.name = "dataset"
ap.round(2)

,GW-CCA (p1p2),SGC,APPNP-lin,Linear ChebNet
dataset,,,,
ogbn-arxiv,97.71,92.35,85.37,93.32
reddit,97.19,86.52,84.27,83.44
yelp,95.35,74.63,67.39,91.05
ogbn-products,99.62,99.66,99.61,NaN
ppi,89.52,86.06,71.22,67.47


## LP AUC gap: GW-CCA − best linear baseline (points)

Positive = GW-CCA is ahead.

In [4]:
best_baseline = auc[["SGC", "APPNP-lin", "Linear ChebNet"]].max(axis=1)
gap = (auc["GW-CCA (p1p2)"] - best_baseline).rename("AUC gap (pt)")
gap.round(2).to_frame()

,AUC gap (pt)
dataset,
ogbn-arxiv,4.48
reddit,9.39
yelp,3.04
ogbn-products,-0.06
ppi,3.95


## Node classification (%)

Metric per dataset: accuracy for multi-class, Micro-F1 for multi-label (yelp, ppi). GW-CCA numbers pulled from the paper's `_res_GCCA__*.json` files (main-paper GW-CCA run).

In [5]:
def _nc(rec):
    return rec.get("nc_test", float("nan"))


rows = {
    "metric": [GCCA[ds].get("metric", "accuracy") for ds in DATASETS],
    "GW-CCA (p1p2)": [GCCA[ds]["test_acc"] * 100 for ds in DATASETS],
    "SGC": [_nc(NC[ds].get("sgc", {})) * 100 if NC[ds].get("sgc") else None for ds in DATASETS],
    "APPNP-lin": [_nc(NC[ds].get("appnp", {})) * 100 if NC[ds].get("appnp") else None for ds in DATASETS],
    "Linear ChebNet": [_nc(NC[ds].get("chebnet", {})) * 100 if NC[ds].get("chebnet") else None for ds in DATASETS],
}
nc = pd.DataFrame(rows, index=DATASETS)
nc.index.name = "dataset"
nc.round(2)

,metric,GW-CCA (p1p2),SGC,APPNP-lin,Linear ChebNet
dataset,,,,,
ogbn-arxiv,accuracy,69.74,70.12,69.62,68.87
reddit,accuracy,96.30,91.42,90.65,94.19
yelp,micro-f1,61.10,23.02,34.96,35.51
ogbn-products,accuracy,72.89,72.21,71.29,NaN
ppi,micro-f1,55.26,52.00,47.82,53.85


## NC gap: GW-CCA − best linear baseline (points)

In [6]:
num_cols = ["SGC", "APPNP-lin", "Linear ChebNet"]
best_nc = nc[num_cols].max(axis=1)
nc_gap = (nc["GW-CCA (p1p2)"] - best_nc).rename("NC gap (pt)")
pd.concat([nc[["metric"]], nc_gap.to_frame()], axis=1).round(2)

,metric,NC gap (pt)
dataset,,
ogbn-arxiv,accuracy,-0.38
reddit,accuracy,2.12
yelp,micro-f1,25.58
ogbn-products,accuracy,0.68
ppi,micro-f1,1.41


## Takeaways for the rebuttal

1. **GW-CCA beats all three linear/diffusion/spectral baselines on 4/5 datasets on LP** (gap +3 to +9 AUC pt) and on 4/5 on NC (gap up to +26 pt on yelp Micro-F1).
2. **ogbn-products LP is a saturation regime.** Every propagator clears 99.5% AUC; GW-CCA is within 0.06 pt of the best. NC still favours GW-CCA.
3. **The three buckets don't collapse to one story.** For LP: SGC best on 3/5, ChebNet on 1/5 (yelp), APPNP-lin never wins. For NC: ChebNet best on reddit / yelp / ppi, SGC on ogbn-arxiv. GW-CCA works uniformly — the CCA-style joint objective, not the propagation operator, is doing the work.
4. **Compare with the M-ablation.** Sweeping $M$ inside GW-CCA moves accuracy by $\pm 0.10$ AUC pt on 4/5 datasets; replacing the *whole* method with a shallow linear/diffusion/spectral propagator drops accuracy by $3$–$9$ pt for LP and up to $26$ pt for NC. The pair-weight matrix is a robust design knob; the CCA objective is load-bearing.